SvasaLens: apnea detection from the nasal thermistor

Finds **when** breathing stops, so the device no longer needs someone else to mark apneas. (The effort
network then decides **which type**.)

- **Data:** PSG-Audio (your Drive: `PSG-Audio/extracted` and `PSG-Audio/rml`).
- **Target:** apneas only (obstructive, central, mixed). A detection on a hypopnea counts neither way.
  False alarms are counted **during sleep** (technologists only score sleep); wake is shown separately.
- **Two detectors:** a simple rule (airflow drops far below normal for long enough) and a small causal
  neural network. Both are tuned without the test patients and tested on the **same held-out patients**
  (the same PSG-Audio test patients as the NN v2 notebook, when its files are found).
- **Decision fixed in advance:** the network is kept only if its F1 is higher than the rule's and the
  95% range of the difference stays above zero, **and** it fits the XIAO.

Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

In [ ]:
# (the two code files are written to Drive by this cell)
SVDET_PY = "\"\"\"SvasaLens apnea DETECTION from the nasal thermistor (finds when breathing stops).\n\nTwo detectors, judged the same way on the same held-out patients:\n  rule    - airflow amplitude drops far below the person's recent normal breathing for long enough\n            (like the hospital AASM rule). Runs forwards in time only, so it can run on the XIAO.\n  network - a small causal neural network that outputs, every second, the chance that an apnea is\n            happening now, followed by the same \"long enough\" step.\n\nScoring follows how a sleep technologist scores: only apneas (obstructive, central, mixed) count as\ntargets; a detection that lands on a hypopnea is neither right nor wrong; false alarms are counted\nduring sleep (wake periods are reported separately, because technologists do not score in wake).\n\"\"\"\nimport re\nimport numpy as np\n\nFS = 8                    # samples per second used by the device\nAPNEA = ('APNEA-O', 'APNEA-C', 'APNEA-MIX')\nASLEEP = ('NonREM1', 'NonREM2', 'NonREM3', 'NonREM4', 'REM')\n\n\n# ----------------------------------------------------------------------------- data\ndef sleep_stages(rml_path, seconds):\n    \"\"\"Per-second True/False 'asleep', from the technologist's (user) staging in the label file.\"\"\"\n    txt = open(rml_path, encoding='utf-8', errors='ignore').read()\n    m = re.search(r'<UserStaging>(.*?)</UserStaging>', txt, re.S)\n    asleep = np.zeros(int(seconds), bool)\n    if not m:\n        return asleep, False\n    st = [(int(float(s)), t) for t, s in re.findall(r'<Stage Type=\"([^\"]+)\" Start=\"([\\d.]+)\"', m.group(1))]\n    st.sort()\n    for k, (s, t) in enumerate(st):\n        e = st[k + 1][0] if k + 1 < len(st) else int(seconds)\n        if t in ASLEEP:\n            asleep[max(0, s):min(int(seconds), e)] = True\n    return asleep, True\n\n\ndef thermistor_8hz(x, rate):\n    from scipy.signal import resample_poly\n    from fractions import Fraction\n    fr = Fraction(FS / rate).limit_denominator(1000)\n    return resample_poly(np.nan_to_num(np.asarray(x, np.float64)), fr.numerator, fr.denominator)\n\n\ndef bandpass(x):\n    \"\"\"0.05-1 Hz, forwards in time only (two biquads, the same kind the chip already runs).\"\"\"\n    from scipy.signal import butter, sosfilt\n    sos = butter(2, [0.05 / (FS / 2), 1.0 / (FS / 2)], btype='band', output='sos')\n    return sosfilt(sos, x)\n\n\ndef load_patient(extracted_npz, rml_path):\n    z = np.load(extracted_npz)\n    import json\n    rates = json.loads(str(z['rates']))\n    seconds = float(z['seconds'])\n    x = bandpass(thermistor_8hz(z['thermistor'], rates['thermistor'])).astype(np.float32)\n    effort = {k: bandpass(thermistor_8hz(z[k], rates[k])).astype(np.float32) for k in ('tho', 'abd') if k in z.files}\n    asleep, staged = sleep_stages(rml_path, int(np.ceil(seconds)))\n    ev = [(float(t), float(d), str(ty)) for (t, d), ty in zip(z['events'], z['types'])]\n    return {'x': x, 'effort': effort, 'asleep': asleep, 'staged': staged, 'seconds': seconds,\n            'apneas': [(t, t + d) for t, d, ty in ev if ty in APNEA],\n            'hyps': [(t, t + d) for t, d, ty in ev if ty == 'HYP']}\n\n\ndef second_labels(p):\n    \"\"\"Per second: 1 = apnea, 0 = normal, -1 = ignore (hypopnea or awake).\"\"\"\n    n = len(p['asleep'])\n    y = np.zeros(n, np.int8)\n    for a, b in p['hyps']:\n        y[int(a):int(np.ceil(b))] = -1\n    for a, b in p['apneas']:\n        y[int(a):int(np.ceil(b))] = 1\n    y[~p['asleep']] = -1\n    return y\n\n\n# ----------------------------------------------------------------------------- shared steps\ndef envelope(x, win_s):\n    \"\"\"Breathing size: root-mean-square of the last win_s seconds, one value per second.\"\"\"\n    w = int(win_s * FS)\n    c = np.concatenate([[0.0], np.cumsum(x.astype(np.float64) ** 2)])\n    n_sec = len(x) // FS\n    ends = (np.arange(n_sec) + 1) * FS\n    starts = np.maximum(0, ends - w)\n    return np.sqrt((c[ends] - c[starts]) / np.maximum(1, ends - starts))\n\n\ndef runs(mask, min_len, merge_gap):\n    \"\"\"Start/end seconds of runs of True at least min_len long; gaps up to merge_gap are bridged.\"\"\"\n    m = np.asarray(mask, bool)\n    if merge_gap > 0 and m.any():\n        idx = np.flatnonzero(m)\n        for a, b in zip(idx[:-1], idx[1:]):\n            if 1 < b - a <= merge_gap + 1:\n                m[a:b] = True\n    out, start = [], None\n    for i, v in enumerate(np.append(m, False)):\n        if v and start is None:\n            start = i\n        elif not v and start is not None:\n            if i - start >= min_len:\n                out.append((start, i))\n            start = None\n    return out\n\n\n# ----------------------------------------------------------------------------- rule\nRULE_DEFAULT = dict(win=6, base_s=120, base_pct=75, drop=0.8, min_dur=10, merge=2, lag=3)\n\n\ndef rule_detect(x, P=RULE_DEFAULT):\n    \"\"\"Apnea = breathing size below (1 - drop) x recent normal size for at least min_dur seconds.\n    'Recent normal size' = base_pct-th percentile of the last base_s seconds (ignores short dips).\"\"\"\n    env = envelope(x, P['win'])\n    n = len(env)\n    base = np.empty(n)\n    k = int(P['base_s'])\n    for i in range(n):\n        lo = max(0, i - k)\n        seg = env[lo:i] if i > lo else env[:1]\n        base[i] = np.percentile(seg, P['base_pct'])\n    low = env < (1 - P['drop']) * np.maximum(base, 1e-9)\n    ev = runs(low, int(P['min_dur']), int(P['merge']))\n    return [(max(0, a - P['lag']), max(0, b - P['lag'])) for a, b in ev]\n\n\ndef rule_baseline_fast(env, base_s, pct):\n    \"\"\"Same baseline as rule_detect, vectorised for tuning speed.\"\"\"\n    from numpy.lib.stride_tricks import sliding_window_view\n    n = len(env); k = int(base_s)\n    pad = np.concatenate([np.full(k, env[0]), env])\n    win = sliding_window_view(pad, k)[:n]          # window i covers env[i-k : i]\n    return np.percentile(win, pct, axis=1)\n\n\ndef rule_detect_fast(env, base, P):\n    low = env < (1 - P['drop']) * np.maximum(base, 1e-9)\n    ev = runs(low, int(P['min_dur']), int(P['merge']))\n    return [(max(0, a - P['lag']), max(0, b - P['lag'])) for a, b in ev]\n\n\n# ----------------------------------------------------------------------------- scoring\ndef score(dets, p):\n    \"\"\"Counts for one patient, with one-to-one matching: each detection can be credited with at most\n    one scored apnea (so one long detection cannot 'find' many apneas). Pairs are matched greedily by\n    largest overlap (at least 1 s). Unmatched detections: 'hyp' if they overlap a hypopnea (not counted\n    either way), otherwise false alarms, split by whether their middle falls in sleep or wake.\"\"\"\n    ap, hy, asleep = p['apneas'], p['hyps'], p['asleep']\n    def olap(a, b, c, d): return min(b, d) - max(a, c)\n    pairs = []\n    for i, (a, b) in enumerate(ap):\n        for j, (c, d) in enumerate(dets):\n            if d < a - 1: continue\n            if c > b + 1: continue\n            o = olap(a, b, c, d)\n            if o >= 1: pairs.append((o, i, j))\n    pairs.sort(reverse=True)\n    used_a, used_d, starts, ends = set(), set(), [], []\n    for o, i, j in pairs:\n        if i in used_a or j in used_d: continue\n        used_a.add(i); used_d.add(j)\n        starts.append(dets[j][0] - ap[i][0]); ends.append(dets[j][1] - ap[i][1])\n    hit = len(used_d); hyp = false_s = false_w = 0\n    for j, (c, d) in enumerate(dets):\n        if j in used_d: continue\n        mid = int((c + d) / 2)\n        sl = bool(asleep[min(mid, len(asleep) - 1)]) if len(asleep) else True\n        if any(olap(a, b, c, d) >= 1 for a, b in hy) or any(olap(a, b, c, d) >= 1 for a, b in ap):\n            hyp += 1          # overlaps a hypopnea, or an apnea already credited to another detection\n        elif sl:\n            false_s += 1\n        else:\n            false_w += 1\n    sleep_h = asleep.sum() / 3600; wake_h = (~asleep).sum() / 3600\n    return dict(apneas=len(ap), found=len(used_a), hit=hit, hyp=hyp, false_sleep=false_s, false_wake=false_w,\n                sleep_h=sleep_h, wake_h=wake_h, ai_true=len(ap) / max(sleep_h, 1e-9),\n                ai_det=(hit + false_s) / max(sleep_h, 1e-9), start_err=starts, end_err=ends)\n\n\ndef summarise(rows):\n    A = sum(r['apneas'] for r in rows); F = sum(r['found'] for r in rows)\n    H = sum(r['hit'] for r in rows); FS_ = sum(r['false_sleep'] for r in rows)\n    FW = sum(r['false_wake'] for r in rows)\n    sh = sum(r['sleep_h'] for r in rows); wh = sum(r['wake_h'] for r in rows)\n    sens = F / max(A, 1); ppv = H / max(H + FS_, 1)\n    f1 = 2 * sens * ppv / max(sens + ppv, 1e-9)\n    se = np.concatenate([r['start_err'] for r in rows]) if A else np.array([])\n    ee = np.concatenate([r['end_err'] for r in rows]) if A else np.array([])\n    t = np.array([r['ai_true'] for r in rows]); d = np.array([r['ai_det'] for r in rows])\n    cat = lambda v: np.digitize(v, [5, 15, 30])\n    return dict(patients=len(rows), apneas=A, sensitivity=sens, ppv=ppv, f1=f1,\n                false_per_sleep_h=FS_ / max(sh, 1e-9), false_per_wake_h=FW / max(wh, 1e-9),\n                start_err_median=float(np.median(np.abs(se))) if len(se) else np.nan,\n                end_err_median=float(np.median(np.abs(ee))) if len(ee) else np.nan,\n                ai_corr=float(np.corrcoef(t, d)[0, 1]) if len(rows) > 2 and t.std() > 0 and d.std() > 0 else np.nan,\n                ai_category_agree=float(np.mean(cat(t) == cat(d))) if len(rows) else np.nan)\n\n\ndef boot_ci(rows, key, n=1000, seed=0):\n    rng = np.random.default_rng(seed); vals = []\n    for _ in range(n):\n        pick = rng.integers(0, len(rows), len(rows))\n        vals.append(summarise([rows[i] for i in pick])[key])\n    return np.percentile(vals, [2.5, 97.5])\n"
SVDETNN_PY = "\"\"\"Small causal neural network for apnea detection (sees only the past, so it can run live on the XIAO).\n\nInput: 8 Hz signals, each divided by the person's recent normal breathing size (same idea as the rule),\nso the network does not depend on one sensor's units. Output: one value per second = chance that an\napnea is happening now.\"\"\"\nimport numpy as np\nimport torch, torch.nn as nn\nimport svdet as D\n\nCH = 16\nDILATIONS = (1, 2, 4, 8, 16, 32, 64)\nK = 9\nRECEPTIVE = 1 + (K - 1) * sum(DILATIONS)          # samples the output can see (about 127 s at 8 Hz)\n\n\ndef normalise(x):\n    \"\"\"Divide by the 90th percentile of the last 120 s of breathing size (past only), clip to +-10.\"\"\"\n    env = D.envelope(x, 6)\n    base = D.rule_baseline_fast(env, 120, 90)\n    b = np.repeat(np.maximum(base, 1e-6), D.FS)[:len(x)]\n    if len(b) < len(x):\n        b = np.concatenate([b, np.full(len(x) - len(b), b[-1] if len(b) else 1.0)])\n    return np.clip(x / b, -10, 10).astype(np.float32)\n\n\nclass CausalConv(nn.Module):\n    def __init__(s, cin, cout, k, d):\n        super().__init__(); s.pad = (k - 1) * d; s.c = nn.Conv1d(cin, cout, k, dilation=d)\n    def forward(s, x): return s.c(nn.functional.pad(x, (s.pad, 0)))\n\n\nclass DetNet(nn.Module):\n    def __init__(s, cin):\n        super().__init__()\n        layers, c = [], cin\n        for d in DILATIONS:\n            layers += [CausalConv(c, CH, K, d), nn.ReLU()]; c = CH\n        s.body = nn.Sequential(*layers); s.head = nn.Conv1d(CH, 1, 1)\n    def forward(s, x):                       # x: (B, C, T) at 8 Hz, T a multiple of 8\n        y = s.head(s.body(x))[:, 0]           # (B, T)\n        return y[:, D.FS - 1::D.FS]           # value at the last sample of each second -> (B, T/8)\n\n\ndef n_params(net): return sum(p.numel() for p in net.parameters())\n\n\ndef macs_per_second(cin):\n    \"\"\"Multiply-adds per second when run live (each layer computes 8 new samples per second).\"\"\"\n    m, c = 0, cin\n    for _ in DILATIONS:\n        m += D.FS * c * CH * K; c = CH\n    return m + CH\n\n\ndef history_bytes(cin, bytes_per_value=4):\n    \"\"\"Memory the live version needs to remember past samples for every layer.\"\"\"\n    tot, c = 0, cin\n    for d in DILATIONS:\n        tot += ((K - 1) * d + 1) * c * bytes_per_value; c = CH\n    return tot\n\n\ndef chunks(pats, chunk_s, n, rng):\n    \"\"\"Random training pieces: (signals, per-second labels). The first RECEPTIVE samples are only context.\"\"\"\n    ctx = int(np.ceil(RECEPTIVE / D.FS))\n    X, Y = [], []\n    lens = np.array([len(p['y']) for p in pats], float)\n    for _ in range(n):\n        p = pats[rng.choice(len(pats), p=lens / lens.sum())]\n        T = len(p['y'])\n        if T <= chunk_s + ctx: continue\n        s0 = rng.integers(0, T - chunk_s - ctx)\n        X.append(p['inp'][:, (s0) * D.FS:(s0 + ctx + chunk_s) * D.FS])\n        y = p['y'][s0:s0 + ctx + chunk_s].astype(np.float32).copy(); y[:ctx] = -1\n        Y.append(y)\n    return np.stack(X), np.stack(Y)\n\n\ndef train(train_pats, tune_pats, cin, dev, seed=0, epochs=30, steps=200, batch=32, chunk_s=300, log=print):\n    torch.manual_seed(seed); rng = np.random.default_rng(seed)\n    net = DetNet(cin).to(dev); opt = torch.optim.Adam(net.parameters(), 1e-3, weight_decay=1e-4)\n    pos = np.mean([np.mean(p['y'][p['y'] >= 0] == 1) for p in train_pats])\n    pw = torch.tensor((1 - pos) / max(pos, 1e-3), device=dev)\n    best, best_state, bad = -1, None, 0\n    for ep in range(epochs):\n        net.train()\n        for _ in range(steps):\n            X, Y = chunks(train_pats, chunk_s, batch, rng)\n            X = torch.tensor(X, device=dev); Y = torch.tensor(Y, device=dev)\n            out = net(X); m = Y >= 0\n            loss = nn.functional.binary_cross_entropy_with_logits(out[m], Y[m], pos_weight=pw)\n            opt.zero_grad(); loss.backward(); opt.step()\n        # tuning score: per-second AUC on tuning patients (sleep, hypopneas ignored)\n        s, l = [], []\n        for p in tune_pats:\n            pr = predict(net, p, dev); m = p['y'][:len(pr)] >= 0\n            s.append(pr[m]); l.append(p['y'][:len(pr)][m])\n        auc = fast_auc(np.concatenate(s), np.concatenate(l) == 1)\n        log(f'    epoch {ep+1}: loss {loss.item():.3f}, tuning AUC {auc:.3f}')\n        if auc > best + 1e-4:\n            best, bad = auc, 0\n            best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}\n        else:\n            bad += 1\n            if bad >= 5: break\n    net.load_state_dict(best_state); return net\n\n\ndef predict(net, p, dev):\n    net.eval()\n    with torch.no_grad():\n        x = torch.tensor(p['inp'][None], device=dev)\n        n = (x.shape[-1] // D.FS) * D.FS\n        return torch.sigmoid(net(x[..., :n]))[0].cpu().numpy()\n\n\ndef detect(prob, thr, min_dur, lag, merge=2):\n    ev = D.runs(prob > thr, int(min_dur), merge)\n    return [(max(0, a - lag), max(0, b - lag)) for a, b in ev]\n\n\ndef fast_auc(s, l):\n    s = np.asarray(s, float); l = np.asarray(l, bool)\n    if l.all() or (~l).all(): return float('nan')\n    o = np.argsort(s, kind='mergesort'); r = np.empty(len(s)); r[o] = np.arange(1, len(s) + 1)\n    n1 = l.sum(); n0 = len(l) - n1\n    return float((r[l].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))\n"
import os, sys, json, glob, time, itertools
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'
PSG = f'{DRIVE}/PSG-Audio'
OUT = f'{DRIVE}/SvasaLens_Detection'
NNV2 = f'{DRIVE}/SvasaLens_NN_v2'          # used only to copy the NN v2 test/tuning patient split
SEED, SPLIT = 0, (0.7, 0.1, 0.2)
os.makedirs(f'{OUT}/code', exist_ok=True)
LOCAL = '/content/work'                   # fast local copy of the data (Drive drops out under heavy reading)
os.makedirs(f'{LOCAL}/cache', exist_ok=True)
for name, text in [('svdet.py', SVDET_PY), ('svdetnn.py', SVDETNN_PY)]:
    open(f'{OUT}/code/{name}', 'w').write(text)
sys.path.insert(0, f'{OUT}/code')
import svdet as D, svdetnn as N
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev)
for sub in ('extracted', 'rml'):
    n = len(glob.glob(f'{PSG}/{sub}/*'))
    print(f'{PSG}/{sub}: {n} files' + ('   <-- MISSING: upload this folder from D:\\files\\PSG-Audio' if n == 0 else ''))

1. Prepare each patient (saved in Drive, so later runs are quick)

In [ ]:
def prepare(pid):
    cf = f'{LOCAL}/cache/{pid}.npz'
    if os.path.exists(cf):
        z = np.load(cf, allow_pickle=True)
        return {'pid': pid, 'x': z['x'].astype(np.float32), 'inp': z['inp'].astype(np.float32), 'y': z['y'],
                'asleep': z['asleep'], 'apneas': [tuple(a) for a in z['apneas']], 'hyps': [tuple(a) for a in z['hyps']],
                'staged': bool(z['staged'])}
    p = D.load_patient(f'{LOCAL}/extracted/{pid}.npz', f'{LOCAL}/rml/{pid}.rml')
    y = D.second_labels(p); n = min(len(p['x']) // D.FS, len(y))
    sig = [p['x']] + [p['effort'][k] for k in ('tho', 'abd') if k in p['effort']]
    inp = np.stack([N.normalise(s)[:n * D.FS] for s in sig])
    np.savez(cf, x=p['x'][:n * D.FS].astype(np.float16), inp=inp.astype(np.float16), y=y[:n], asleep=p['asleep'][:n],
             apneas=np.array(p['apneas'], float).reshape(-1, 2), hyps=np.array(p['hyps'], float).reshape(-1, 2),
             staged=p['staged'])
    return prepare(pid)

# label files (.rml) may sit in PSG-Audio/rml or in a subfolder of it (e.g. rml/rml after unzipping)
all_ext = sorted(os.path.basename(f)[:-4] for f in glob.glob(f'{PSG}/extracted/*.npz'))
RML = {}
for f in glob.glob(f'{PSG}/**/*.rml', recursive=True):
    RML.setdefault(os.path.basename(f)[:-4], f)
pids = [p for p in all_ext if p in RML]
print(f'signal files: {len(all_ext)}, label files found: {len(RML)}, patients with both: {len(pids)}')
if len(pids) < 50:
    print('Example signal names:', all_ext[:3], ' example label names:', list(RML)[:3])
    raise SystemExit('STOP: fewer than 50 patients have both files. Is the rml upload to Drive finished? '
                     'Send Claude the lines above.')
# Copy every patient's two files from Drive to this machine first (one file at a time, reconnecting
# to Drive if it drops), then work only on the local copies.
import shutil
def remount():
    print('\n  Drive connection dropped - reconnecting...')
    try: drive.mount('/content/drive', force_remount=True)
    except Exception as e: print('  reconnect error:', e)
    time.sleep(3)

def copy_in(src, dst):
    if os.path.exists(dst) and os.path.getsize(dst) > 0: return True
    for attempt in range(6):
        try:
            shutil.copyfile(src, dst + '.part'); os.replace(dst + '.part', dst); return True
        except OSError:
            remount()
    return False

os.makedirs(f'{LOCAL}/extracted', exist_ok=True); os.makedirs(f'{LOCAL}/rml', exist_ok=True)
t0 = time.time(); not_copied = []
for k, pid in enumerate(pids):
    ok = copy_in(f'{PSG}/extracted/{pid}.npz', f'{LOCAL}/extracted/{pid}.npz') and copy_in(RML[pid], f'{LOCAL}/rml/{pid}.rml')
    if not ok: not_copied.append(pid)
    print(f'\r  copying from Drive: {k+1}/{len(pids)} patients ({time.time()-t0:.0f} s)', end='')
pids = [p for p in pids if p not in not_copied]
print(f'\ncopied {len(pids)} patients' + (f'; could not copy {len(not_copied)}: {not_copied[:5]}' if not_copied else ''))

PAT, skipped = {}, []
t0 = time.time()
for k, pid in enumerate(pids):
    try:
        p = prepare(pid)
        if not p['staged'] or p['asleep'].sum() < 3600: skipped.append((pid, 'no sleep staging or < 1 h sleep')); continue
        PAT[pid] = p
    except Exception as e:
        skipped.append((pid, str(e)[:80]))
    print(f'\r  {k+1}/{len(pids)} patients  ({time.time()-t0:.0f} s)', end='')
if len(PAT) < 50:
    from collections import Counter
    print('\nreasons patients were left out:', Counter(r for _, r in skipped).most_common(5))
    raise SystemExit(f'STOP: only {len(PAT)} patients usable. Send Claude the lines above.')
print(f'\nready: {len(PAT)} patients, {sum(len(p["apneas"]) for p in PAT.values())} apneas, '
      f'{sum(p["asleep"].sum() for p in PAT.values())/3600:.0f} h of sleep')
for s in skipped: print('  left out:', s)

2. Split patients (same test/tuning patients as NN v2 where possible)

In [ ]:
def nnv2_patient_list():
    fs = sorted(glob.glob(f'{NNV2}/psgaudio_*patients_events.npz') + glob.glob(f'{DRIVE}/*/psgaudio_*patients_events.npz'), key=lambda f: int(f.split('_')[-2].replace('patients', '')))
    if not fs: return None, None
    return np.array(sorted(np.unique(np.load(fs[-1])['G']))), fs[-1]

import re
def key(name):                      # patient number only, e.g. '00001010-100507.npz' -> '1010'
    m = re.search(r'\d+', str(name)); return m.group().lstrip('0') if m else str(name)
BY_KEY = {key(p): p for p in PAT}

def make_split(pats):
    rng = np.random.default_rng(SEED); pats = np.array(sorted(pats)); rng.shuffle(pats)
    a = int(SPLIT[0] * len(pats)); b = a + int(SPLIT[1] * len(pats))
    tune = [BY_KEY[key(p)] for p in pats[a:b] if key(p) in BY_KEY]
    test = [BY_KEY[key(p)] for p in pats[b:] if key(p) in BY_KEY]
    return tune, test

pats, src = nnv2_patient_list()
TUNE, TEST = make_split(pats) if pats is not None else ([], [])
print('NN v2 patient list:', src, '|', 'not found' if pats is None else
      f'{len(pats)} patients, e.g. {list(pats[:3])}; matched here: {sum(key(p) in BY_KEY for p in pats)}')
if len(TUNE) < 5 or len(TEST) < 10:
    TUNE, TEST = make_split(list(PAT))
    src = 'own split of the patients found here (NN v2 split could not be matched)'
TRAIN = [p for p in PAT if p not in set(TUNE) | set(TEST)]
assert TUNE and TEST, 'no tuning or test patients - send Claude the output above'
print(f'split from: {src}')
print(f'train {len(TRAIN)}, tuning {len(TUNE)}, test {len(TEST)} patients '
      f'(test apneas: {sum(len(PAT[p]["apneas"]) for p in TEST)})')

3. Rule detector: tune on training + tuning patients, then test once

In [ ]:
GRID = dict(win=(4, 6, 8), base_s=(60, 120), base_pct=(75, 90), drop=(0.3, 0.4, 0.5, 0.6), min_dur=(6, 8, 10))
fit = [PAT[p] for p in TRAIN + TUNE]
t0 = time.time(); results = []
for win in GRID['win']:
    envs = [D.envelope(p['x'], win) for p in fit]
    for bs, pct in itertools.product(GRID['base_s'], GRID['base_pct']):
        bases = [D.rule_baseline_fast(e, bs, pct) for e in envs]
        for drop, md in itertools.product(GRID['drop'], GRID['min_dur']):
            P_ = dict(win=win, base_s=bs, base_pct=pct, drop=drop, min_dur=md, merge=2, lag=win // 2)
            rows = [D.score(D.rule_detect_fast(e, b_, P_), p) for e, b_, p in zip(envs, bases, fit)]
            results.append((D.summarise(rows)['f1'], P_))
    print(f'  window {win} s done ({time.time()-t0:.0f} s)')
results.sort(key=lambda r: -r[0])
RULE = results[0][1]
print('best rule settings (chosen without test patients):', RULE, f' F1 on fitting patients {results[0][0]:.3f}')

def rule_rows(pids_):
    return [D.score(D.rule_detect(PAT[p]['x'], RULE), PAT[p]) for p in pids_]
RULE_TEST = rule_rows(TEST)

4. Network detectors: train, tune the alarm settings on tuning patients, then test once

In [ ]:
REUSE = True    # True: load a network already trained by an earlier run instead of training again

def nn_detector(name, ch):
    f = f'{OUT}/{name}.pt'
    if REUSE and os.path.exists(f):
        ck = torch.load(f, map_location=dev); net = N.DetNet(len(ch)).to(dev); net.load_state_dict(ck['weights'])
        print(f'{name}: loaded'); return net, ck['alarm']
    net = N.train([PAT[p] for p in TRAIN], [PAT[p] for p in TUNE], len(ch), dev, seed=SEED)
    probs = {p: N.predict(net, PAT[p], dev) for p in TUNE}
    best = (-1, None)
    for thr, md, lag in itertools.product(np.arange(0.3, 0.95, 0.05), (4, 6, 8, 10), (0, 2, 4)):
        rows = [D.score(N.detect(probs[p], thr, md, lag), PAT[p]) for p in TUNE]
        f1 = D.summarise(rows)['f1']
        if f1 > best[0]: best = (f1, dict(thr=float(thr), min_dur=md, lag=lag))
    print(f'{name}: alarm settings (from tuning patients) {best[1]}, tuning F1 {best[0]:.3f}')
    torch.save({'weights': net.state_dict(), 'alarm': best[1], 'channels': ch}, f)
    return net, best[1]

for p in PAT.values(): p['_inp_all'] = p['inp']
NETS = {}
for name, ch in (('nn_thermistor', [0]), ('nn_thermistor_effort', [0, 1, 2])):
    for p in PAT.values(): p['inp'] = p['_inp_all'][ch]
    net, alarm = nn_detector(name, ch)
    rows = [D.score(N.detect(N.predict(net, PAT[p], dev), alarm['thr'], alarm['min_dur'], alarm['lag']), PAT[p]) for p in TEST]
    NETS[name] = (net, alarm, ch, rows)
for p in PAT.values(): p['inp'] = p['_inp_all']

5. Results on the held-out test patients

In [ ]:
def line(name, rows):
    s = D.summarise(rows); lo, hi = D.boot_ci(rows, 'f1'); sl, sh = D.boot_ci(rows, 'sensitivity')
    return (f"  {name:28s} {100*s['sensitivity']:6.1f}% ({100*sl:.0f}-{100*sh:.0f})  {100*s['ppv']:6.1f}%  "
            f"{s['f1']:.3f} ({lo:.3f}-{hi:.3f})  {s['false_per_sleep_h']:6.1f}  {s['false_per_wake_h']:6.1f}  "
            f"{s['start_err_median']:5.1f}/{s['end_err_median']:<5.1f}  {s['ai_corr']:.2f}  {100*s['ai_category_agree']:5.1f}%")

def paired_diff(rows_a, rows_b, key='f1', n=2000):
    rng = np.random.default_rng(SEED); d = []
    for _ in range(n):
        i = rng.integers(0, len(rows_a), len(rows_a))
        d.append(D.summarise([rows_a[k] for k in i])[key] - D.summarise([rows_b[k] for k in i])[key])
    return np.percentile(d, [2.5, 97.5])

L = [f'SvasaLens apnea detection from the thermistor  ({time.strftime("%Y-%m-%d %H:%M")}, seed {SEED})',
     f'PSG-Audio held-out test patients: {len(TEST)} ({sum(len(PAT[p]["apneas"]) for p in TEST)} scored apneas). '
     f'Settings chosen without them.', '',
     f"  {'detector':28s} {'found':>15s}  {'right':>7s}  {'F1 (95% range)':>19s}  {'false/h':>6s}  {'false/h':>6s}  "
     f"{'timing s':>11s}  {'AI r':>4s}  {'AI cat':>6s}",
     f"  {'':28s} {'(sensitivity)':>15s}  {'(PPV)':>7s}  {'':19s}  {'sleep':>6s}  {'wake':>6s}  {'start/end':>11s}",
     line('rule (thermistor)', RULE_TEST)]
for name, (net, alarm, ch, rows) in NETS.items():
    L.append(line(name.replace('_', ' '), rows))
L += ['', 'found = share of scored apneas detected; right = share of alarms (in sleep, not on a hypopnea) that were real apneas;',
      'false/h = false alarms per hour of sleep / of wake; timing = median start/end error (s) of found apneas;',
      'AI r = correlation of detected vs scored apneas per hour of sleep (per patient); AI cat = same severity group (<5, 5-15, 15-30, 30+).', '']
L.append('Rule settings: ' + json.dumps(RULE))
KEEP = None
for name, (net, alarm, ch, rows) in NETS.items():
    lo, hi = paired_diff(rows, RULE_TEST)
    better = lo > 0
    L.append(f'{name}: F1 minus rule F1, 95% range {lo:+.3f} to {hi:+.3f} -> '
             + ('clearly better than the rule' if better else 'NOT clearly better than the rule'))
    if better and (KEEP is None or D.summarise(rows)['f1'] > D.summarise(NETS[KEEP][3])['f1']):
        KEEP = name
print('\n'.join(L))

6. Can the XIAO run it?

In [ ]:
# ESP32-S3 on the XIAO Sense: 240 MHz dual core, 512 KB internal RAM, 8 MB PSRAM.
# Live, the network computes 8 new samples per second per layer and remembers a short history per layer.
X = ['', 'XIAO ESP32-S3 check (live, one new result per second):']
for name, (net, alarm, ch, rows) in NETS.items():
    prm = N.n_params(net); macs = N.macs_per_second(len(ch)); hist = N.history_bytes(len(ch))
    # very conservative speed for plain C on one core, no vector instructions: about 20 million multiply-adds/s
    load = macs / 20e6
    X.append(f'  {name}: {prm:,} weights ({prm*4/1024:.0f} KB as float, {prm/1024:.0f} KB as int8), '
             f'{macs:,} multiply-adds per second ({100*load:.2f}% of one core, conservative), '
             f'history memory {hist/1024:.0f} KB  -> fits easily')
X.append(f'  rule: a few hundred bytes, trivial.')
X.append('')
X.append('DECISION (fixed in advance): ' + (f'use the network "{KEEP}" for detection.' if KEEP else
         'keep the RULE: no network was clearly better than it on the held-out test patients.'))
print('\n'.join(X))
open(f'{OUT}/detection_results.txt', 'w').write('\n'.join(L + X) + '\n')
json.dump({'rule': RULE, 'keep': KEEP, 'nets': {k: v[1] for k, v in NETS.items()}}, open(f'{OUT}/detection_settings.json', 'w'))
print(f'\nsaved {OUT}/detection_results.txt')